# Day 054 — Exercise 4: Persistence & Migrations

**What you'll build:** `make_engine(db_path)` (a **file-backed** engine whose data survives a restart), `column_exists(engine, table, column)`, and `migrate_add_column(engine, table, column, sqltype)` — a minimal, idempotent migration.

**Why it matters:** In-memory data vanishes when the process ends. A **file-backed** database is what gives the app *saved state*. And as the app evolves, its schema changes — a migration adds a column to an existing database without destroying the data already in it.

## Provided: Setup + Models + repository functions

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import tempfile
from datetime import datetime
from sqlalchemy import create_engine, ForeignKey, select, inspect as sa_inspect, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Conversation(Base):
    """One chat conversation. Has many Messages (one-to-many)."""
    __tablename__ = 'conversations'

    id:         Mapped[int]      = mapped_column(primary_key=True)
    title:      Mapped[str]      = mapped_column(default='New chat')
    created_at: Mapped[datetime] = mapped_column(default=datetime.utcnow)

    # relationship() is the ORM link (not a DB column). cascade deletes a
    # conversation's messages when the conversation is deleted.
    messages: Mapped[list['Message']] = relationship(
        back_populates='conversation', cascade='all, delete-orphan')


class Message(Base):
    """One message in a conversation. Belongs to one Conversation (many-to-one)."""
    __tablename__ = 'messages'

    id:              Mapped[int]      = mapped_column(primary_key=True)
    conversation_id: Mapped[int]      = mapped_column(ForeignKey('conversations.id'))
    role:            Mapped[str]      = mapped_column()
    content:         Mapped[str]      = mapped_column()
    created_at:      Mapped[datetime] = mapped_column(default=datetime.utcnow)

    conversation: Mapped['Conversation'] = relationship(back_populates='messages')


def memory_engine():
    """In-memory SQLite engine for tests. StaticPool makes every Session share the
    one in-memory database (see Day 44)."""
    return create_engine('sqlite:///:memory:',
                          connect_args={'check_same_thread': False},
                          poolclass=StaticPool)


def create_schema(engine) -> None:
    """Create every table registered on Base (CREATE TABLE IF NOT EXISTS)."""
    Base.metadata.create_all(engine)


def create_conversation(session, title: str = 'New chat') -> Conversation:
    """Insert a new conversation and flush so its auto id is assigned.
    The caller controls commit (unit-of-work pattern)."""
    conv = Conversation(title=title)
    session.add(conv)
    session.flush()
    return conv


def add_message(session, conversation_id: int, role: str, content: str) -> Message:
    """Append a message to a conversation via its foreign key, and flush to assign
    the id. The caller commits."""
    msg = Message(conversation_id=conversation_id, role=role, content=content)
    session.add(msg)
    session.flush()
    return msg


def get_messages(session, conversation_id: int) -> list:
    """Return the conversation's messages, in insertion order, as plain
    [{'role', 'content'}] dicts (safe to use after the session closes)."""
    stmt = (select(Message)
            .where(Message.conversation_id == conversation_id)
            .order_by(Message.id))
    rows = session.execute(stmt).scalars().all()
    return [{'role': m.role, 'content': m.content} for m in rows]


def list_conversations(session) -> list:
    """Return all conversations as [{'id', 'title', 'message_count'}] by id."""
    convs = session.execute(
        select(Conversation).order_by(Conversation.id)).scalars().all()
    return [{'id': c.id, 'title': c.title, 'message_count': len(c.messages)}
            for c in convs]

## Your Implementation

In [ ]:
def make_engine(db_path: str):
    """File-backed SQLite engine; create the schema. Data survives restarts."""
    # TODO: engine = create_engine(f'sqlite:///{db_path}')
    # TODO: Base.metadata.create_all(engine)
    # TODO: return engine
    pass


def column_exists(engine, table: str, column: str) -> bool:
    """True if `column` already exists on `table`."""
    # TODO: return column in [c['name'] for c in sa_inspect(engine).get_columns(table)]
    pass


def migrate_add_column(engine, table: str, column: str, sqltype: str = 'TEXT') -> bool:
    """Add the column only if missing. Return True if added, False if it existed."""
    # TODO: if column_exists(engine, table, column): return False
    # TODO: with engine.begin() as conn:
    #     conn.execute(text(f'ALTER TABLE {table} ADD COLUMN {column} {sqltype}'))
    # TODO: return True
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    db_dir = tempfile.mkdtemp()
    db_path = os.path.join(db_dir, 'app.db')

    # Check 1: make_engine creates a real file with the schema
    try:
        engine = make_engine(db_path)
        assert os.path.exists(db_path), 'database file was not created'
        assert {'conversations', 'messages'} <= set(sa_inspect(engine).get_table_names())
        passed += 1; print('✅ Check 1: make_engine creates a file-backed schema')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: data SURVIVES a restart (new engine, same file)
    try:
        with Session(engine) as s:
            conv = create_conversation(s, 'persisted')
            add_message(s, conv.id, 'user', 'remember me')
            s.commit()
            cid = conv.id
        engine.dispose()                      # simulate shutdown
        engine2 = make_engine(db_path)        # simulate restart
        with Session(engine2) as s:
            msgs = get_messages(s, cid)
        assert msgs == [{'role': 'user', 'content': 'remember me'}], f'data lost: {msgs}'
        passed += 1; print('✅ Check 2: data survives a restart (file-backed)')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: column_exists reports existing vs missing columns
    try:
        assert column_exists(engine2, 'conversations', 'title') is True
        assert column_exists(engine2, 'conversations', 'pinned') is False
        passed += 1; print('✅ Check 3: column_exists is accurate')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: migrate_add_column adds a new column
    try:
        added = migrate_add_column(engine2, 'conversations', 'pinned', 'INTEGER')
        assert added is True, 'first migration should add the column (True)'
        assert column_exists(engine2, 'conversations', 'pinned') is True
        passed += 1; print('✅ Check 4: migrate_add_column adds the column')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: the migration is idempotent (safe to re-run)
    try:
        again = migrate_add_column(engine2, 'conversations', 'pinned', 'INTEGER')
        assert again is False, 'second run should be a no-op (False)'
        passed += 1; print('✅ Check 5: migration is idempotent')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def make_engine(db_path: str):
    """File-backed SQLite engine — data SURVIVES a process restart. Creates the
    schema on first use (safe to call every startup)."""
    engine = create_engine(f'sqlite:///{db_path}')
    Base.metadata.create_all(engine)
    return engine


def column_exists(engine, table: str, column: str) -> bool:
    """True if `column` already exists on `table` (schema introspection)."""
    return column in [c['name'] for c in sa_inspect(engine).get_columns(table)]


def migrate_add_column(engine, table: str, column: str, sqltype: str = 'TEXT') -> bool:
    """A minimal, idempotent migration: add a column only if it is missing.
    Returns True if it added the column, False if it was already there.
    Safe to run on every startup — this is the essence of a migration."""
    if column_exists(engine, table, column):
        return False
    with engine.begin() as conn:
        conn.execute(text(f'ALTER TABLE {table} ADD COLUMN {column} {sqltype}'))
    return True
```

**Why this works:** `sqlite:///{db_path}` points the engine at a file instead of memory, so the data is on disk and outlives the process — that is *saved state*. `column_exists` introspects the live schema. `migrate_add_column` checks first and only issues `ALTER TABLE ADD COLUMN` when needed, so running it on every startup is safe: it upgrades an old database once and does nothing thereafter. That check-then-change pattern is the heart of every migration (production apps use Alembic to manage many of them in order).
</details>